In [13]:
import pandas as pd

catalogue = pd.read_csv("../catalogue/catalogue.csv")
print(catalogue.shape)
catalogue.head()

(3323, 27)


,tconst,titre_international,titre_affiche,annee_sortie,duree_min,genres,note_moyenne,nombre_votes,original_language,production_countries,...,emo_rire,emo_frissonner,emo_emouvoir,emo_evader,emo_reflechir,emo_vibrer,acteurs_y,realisateurs_y,acteurs,realisateurs
0,tt0006864,Intolerance,Intolérance,1916,210,"Drama,History",7.6,18277,en,United States of America,...,False,False,False,False,True,False,"Lillian Gish, Robert Harron, Mae Marsh",D.W. Griffith,"Lillian Gish, Robert Harron, Mae Marsh",D.W. Griffith
1,tt0010323,The Cabinet of Dr. Caligari,Le Cabinet du docteur Caligari,1920,67,"Horror,Mystery,Thriller",8.0,77573,de,Germany,...,False,True,False,False,False,False,"Werner Krauss, Conrad Veidt, Friedrich Feher",Robert Wiene,"Werner Krauss, Conrad Veidt, Friedrich Feher",Robert Wiene
2,tt0012349,The Kid,Le Kid,1921,68,"Comedy,Drama,Family",8.2,147895,en,United States of America,...,True,False,False,True,False,False,"Charles Chaplin, Edna Purviance, Jackie Coogan",NaN,"Charles Chaplin, Edna Purviance, Jackie Coogan",NaN
3,tt0012364,The Phantom Carriage,La charrette fantôme,1921,107,"Drama,Fantasy,Horror",8.0,16201,sv,Sweden,...,False,True,False,True,False,False,"Victor Sjöström, Hilda Borgström, Tore Svennberg",NaN,"Victor Sjöström, Hilda Borgström, Tore Svennberg",NaN
4,tt0012494,Destiny,Les Trois Lumières,1921,114,"Drama,Fantasy,Horror",7.6,7525,de,Germany,...,False,True,False,True,False,False,"Bernhard Goetzke, Lil Dagover, Walter Janssen",Fritz Lang,"Bernhard Goetzke, Lil Dagover, Walter Janssen",Fritz Lang


In [14]:
emotion = "emo_frissonner"   # à changer pour tester une autre émotion, voir par emotion le catalogue

films_possibles = catalogue[catalogue[emotion] == True]
print(len(films_possibles), "films possibles")

film = films_possibles.sample(1)
film[["titre_affiche", "annee_sortie", "note_moyenne"]]

1002 films possibles


,titre_affiche,annee_sortie,note_moyenne
2807,Ugramm,2014,8.0


In [15]:
# un film peut avoir plusieurs émotions donc plus de 3323, normal
emotions = ["emo_emouvoir", "emo_evader", "emo_frissonner", "emo_reflechir", "emo_rire", "emo_vibrer"]
catalogue[emotions].sum().sort_values(ascending=False)

emo_rire          1070
emo_emouvoir      1019
emo_frissonner    1002
emo_vibrer         785
emo_evader         727
emo_reflechir      699
dtype: int64

In [16]:
catalogue["genres"].head()

0              Drama,History
1    Horror,Mystery,Thriller
2        Comedy,Drama,Family
3       Drama,Fantasy,Horror
4       Drama,Fantasy,Horror
Name: genres, dtype: str

In [17]:
genres_01 = catalogue["genres"].str.get_dummies(sep=",")
print(genres_01.shape)
genres_01.head()

(3323, 23)


,Action,Adventure,Animation,Biography,Comedy,Crime,Documentary,Drama,Family,Fantasy,...,Music,Musical,Mystery,News,Romance,Sci-Fi,Sport,Thriller,War,Western
0,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,1,0,0
2,0,0,0,0,1,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0


Le plan:

Préparer les caractéristiques (features) : transformer genres en colonnes 0/1 et mettre note, année et durée à la même échelle.
Découper en train/test avec train_test_split(test_size=0.2).
Entraîner un KNeighborsClassifier et mesurer son score.
Faire la fonction finale : film aimé + émotion donnent les 5 films voisins les plus proches qui ont cette émotion. 5 ou 1, ce sera juste un paramètre, donc vous déciderez plus tard.

In [18]:
from sklearn.preprocessing import MinMaxScaler

# 1. On garde les 3 colonnes numériques qui décrivent un film
num = catalogue[["note_moyenne", "annee_sortie", "duree_min"]]

# 2. Mise à l'échelle : MinMaxScaler ramène chaque colonne entre 0 et 1
#    (année 1920 -> 0, année 2026 -> 1 ; durée la plus courte -> 0, la plus longue -> 1)
#    Sans ça, l'année (des milliers) écraserait les genres (0 ou 1) dans le calcul des distances
num_scaled = pd.DataFrame(MinMaxScaler().fit_transform(num), columns=num.columns)

# 3. Concaténation : on colle côte à côte les 23 colonnes de genres (0/1)
#    et les 3 colonnes numériques (0 à 1) -> X = 26 caractéristiques par film
X = pd.concat([genres_01, num_scaled], axis=1)

# 4. y = ce que le modèle doit prédire : les 6 émotions, True/False converties en 1/0
y = catalogue[emotions].astype(int)

# 5. Vérification : 3323 films, 26 caractéristiques (X) et 6 émotions (y)
print(X.shape, y.shape)

(3323, 26) (3323, 6)


In [19]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
print("Score :", knn.score(X_test, y_test))

#0.94 score élevé du modèle car les emotions sont déduites des genres

Score : 0.9488721804511279


In [27]:
from sklearn.neighbors import NearestNeighbors

def recommander(emotion1, emotion2=None, n=5):
    # on garde les films de l'émotion choisie (et de la 2e si elle est donnée)
    masque = catalogue[emotion1] == True
    if emotion2:
        masque = masque & (catalogue[emotion2] == True)
    X_emo = X[masque]
    films_emo = catalogue[masque]

    # profil "idéal" : la moyenne des films de cette émotion
    profil = X_emo.mean().to_frame().T

    # on cherche les 20 plus proches, puis on en tire n au hasard -> variété à chaque appel
    nn = NearestNeighbors(n_neighbors=min(20, len(X_emo)))
    nn.fit(X_emo)
    distances, indices = nn.kneighbors(profil)
    proches = films_emo.iloc[indices[0]]
    return proches.sample(min(n, len(proches)))[["titre_affiche", "annee_sortie", "note_moyenne", "genres"]]

recommander("emo_frissonner")

,titre_affiche,annee_sortie,note_moyenne,genres
1752,Vera Drake,2004,7.6,"Crime,Drama"
1247,La dernière marche,1995,7.5,"Crime,Drama"
1897,This is England,2006,7.7,"Crime,Drama"
1609,Carandiru,2003,7.6,"Crime,Drama"
1220,L'âme des guerriers,1994,7.9,"Crime,Drama"
